In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("one_big_table_publico")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c9368fc0-8124-45ee-915e-0bf838917c75;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 146ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://silver/base_score_bureau_movel/"
df_base_score_bureau_movel = spark.read.parquet(path)
df_base_score_bureau_movel.show(5, truncate=False)

25/12/31 11:17:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+-----------+---------------+---+----+---------+--------+--------+--------------+---+------+
|NUM_CPF    |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|DATPROC       |rn |SAFRA |
+-----------+---------------+---+----+---------+--------+--------+--------------+---+------+
|777799XN7XT|1              |0  |CMV |PRE      |570     |768     |20251230073052|1  |202412|
|7777N8XZY8Y|1              |0  |CMV |PRE      |564     |579     |20251230073052|1  |202412|
|7777W9777ZU|1              |0  |CMV |PRE      |677     |758     |20251230073052|1  |202412|
|7777WXNZUU9|1              |1  |CMV |PRE      |555     |541     |20251230073052|1  |202412|
|7777XZTYTN8|1              |0  |CMV |PRE      |481     |652     |20251230073052|1  |202412|
+-----------+---------------+---+----+---------+--------+--------+--------------+---+------+
only showing top 5 rows



In [4]:
df_base_score_bureau_movel.count()

1290526

In [5]:
path = "s3a://silver/base_telco/"
df_base_telco = spark.read.parquet(path)
df_base_telco.show(5, truncate=False)

25/12/31 11:17:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+--------------+------+
|NUM_CPF    |FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72|var_73|var_74|var_75|var_76|var_77|var_78|var_79|var_80|var_81|var_82|var_83

In [6]:
path = "s3a://silver/base_dados_cadastrais/"
df_base_dados_cadastrais = spark.read.parquet(path)
df_base_dados_cadastrais.show(5, truncate=False)

+-----------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+------+------+------+------+------+------+------+------+------------+------+------+--------+-----------------------+-------------+--------------+------+
|NUM_CPF    |FLAG_INSTALACAO|FPD |PROD|flag_mig2|STATUSRF|DATADENASCIMENTO|var_03|var_02|var_04|var_05|var_06|var_07|var_08|var_09|var_10|var_11|var_12    |var_13|var_14|var_15|var_16|var_17|var_18|var_19|var_20|var_21      |var_22|var_23|var_24  |var_25                 |CEP_3_digitos|DATPROC       |SAFRA |
+-----------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+------+------+------+------+------+------+------+------+------------+------+------+--------+-----------------------+-------------+--------------+------+
|7WX9UZ9W8ZZ|1              |0   |CMV |Aquisição|REGULAR |1952-07-04     

In [7]:
df_base_score_bureau_movel.createOrReplaceTempView("bureau")
df_base_telco.createOrReplaceTempView("telco")
df_base_dados_cadastrais.createOrReplaceTempView("cadastral")


In [8]:
score_cols = set(df_base_score_bureau_movel.columns)
telco_cols = set(df_base_telco.columns) - score_cols
cad_cols   = set(df_base_dados_cadastrais.columns) - score_cols


In [9]:
telco_select = ",\n    ".join([f"t.{c}" for c in telco_cols])
cad_select   = ",\n    ".join([f"c.{c}" for c in cad_cols])

query = f"""
SELECT
    b.*,
    {telco_select},
    {cad_select}
FROM bureau b
LEFT JOIN telco t
    ON b.NUM_CPF = t.NUM_CPF AND b.SAFRA  = t.SAFRA 
LEFT JOIN cadastral c
    ON b.NUM_CPF = c.NUM_CPF AND b.SAFRA  = c.SAFRA 
"""
df_final = spark.sql(query)


In [10]:
df_final.createOrReplaceTempView("df_final")
df_final.cache()
df_final.count()

1290526

In [11]:
df_final.show(5, truncate=False)

+-----------+---------------+---+----+---------+--------+--------+--------------+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------------+-------------+-------------+------+----------------+------+-------------------------+------+------+------+------+------------+------+------+----------+------+----------+------+------+------+----------+------+------+--------+
|NUM_CPF    |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|DATPROC       |rn |SAFRA |var_37|var_28|var_92|var_51|var_50|var_81|var_48|var_88|var_46|var_27|var_70|var_

In [16]:
print('lista de colunas para tipar')
for col in spark.table("df_final").columns:
    print(f"{col},")

lista de colunas para tipar
NUM_CPF,
FLAG_INSTALACAO,
FPD,
PROD,
flag_mig2,
SCORE_01,
SCORE_02,
DATPROC,
rn,
SAFRA,
var_37,
var_28,
var_92,
var_51,
var_50,
var_81,
var_48,
var_88,
var_46,
var_27,
var_70,
var_87,
var_26,
var_40,
var_30,
var_62,
var_36,
var_56,
var_39,
var_76,
var_49,
var_38,
var_68,
var_52,
var_89,
var_64,
var_33,
var_82,
var_66,
var_69,
var_84,
var_61,
var_67,
var_43,
var_32,
var_75,
var_65,
var_83,
var_78,
var_47,
var_79,
var_44,
var_90,
var_57,
var_58,
var_35,
var_42,
var_29,
var_72,
var_31,
var_73,
var_53,
var_34,
var_71,
var_80,
var_63,
var_86,
var_85,
var_93,
var_54,
var_45,
var_74,
var_55,
var_41,
var_77,
var_91,
var_59,
var_60,
var_17,
var_11,
var_04,
var_22,
CEP_3_digitos,
var_23,
var_09,
DATADENASCIMENTO,
var_15,
var_25,
var_14,
var_02,
var_18,
var_20,
var_21,
var_16,
var_19,
var_13,
var_03,
var_12,
var_07,
var_06,
var_05,
var_24,
var_10,
var_08,
STATUSRF,


In [22]:
one_big_table = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            NUM_CPF,
            SAFRA,
            FLAG_INSTALACAO,
            FPD,
            PROD,
            flag_mig2,
            SCORE_01,
            SCORE_02,
            rn,            
            var_37,
            var_28,
            var_92,
            var_51,
            var_50,
            var_81,
            var_48,
            var_88,
            var_46,
            var_27,
            var_70,
            var_87,
            var_26,
            var_40,
            var_30,
            var_62,
            var_36,
            var_56,
            var_39,
            var_76,
            var_49,
            var_38,
            var_68,
            var_52,
            var_89,
            var_64,
            var_33,
            var_82,
            var_66,
            var_69,
            var_84,
            var_61,
            var_67,
            var_43,
            var_32,
            var_75,
            var_65,
            var_83,
            var_78,
            var_47,
            var_79,
            var_44,
            var_90,
            var_57,
            var_58,
            var_35,
            var_42,
            var_29,
            var_72,
            var_31,
            var_73,
            var_53,
            var_34,
            var_71,
            var_80,
            var_63,
            var_86,
            var_85,
            var_93,
            var_54,
            var_45,
            var_74,
            var_55,
            var_41,
            var_77,
            var_91,
            var_59,
            var_60,
            var_17,
            var_11,
            var_04,
            var_22,
            CEP_3_digitos,
            var_23,
            var_09,
            case 
                when trim(DATADENASCIMENTO) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else cast(
                    cast(SAFRA / 100 as int) 
                    - year(to_date(trim(DATADENASCIMENTO), 'yyyy-MM-dd'))
                    as int
                )
            end as IDADE,
            var_15,
            var_25,
            var_14,
            var_02,
            var_18,
            var_20,
            var_21,
            var_16,
            var_19,
            var_13,
            var_03,
            var_12,
            var_07,
            var_06,
            var_05,
            var_24,
            var_10,
            var_08,
            STATUSRF,
            {pdthproc} as DATPROC

        from
            df_final
        order by
            NUM_CPF,
            SAFRA
            
    """.format(pdthproc=dthproc))
one_big_table.createOrReplaceTempView("one_big_table")
one_big_table.cache()
one_big_table.count()  

1290526

In [23]:
one_big_table.show(5, truncate=False)

+-----------+------+---------------+---+----+---------+--------+--------+---+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------------+-------------+------+------+-----+------+-------------------------+------+------+------+------+------------+------+------+----------+------+----------+------+------+------+----------+------+------+--------+--------------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|SCORE_01|SCORE_02|rn |var_37|var_28|var_92|var_51|var_50|var_81|var_48|var_88|var_46|var_27|var_70|var_87|var_26|var_40|var_30|var_62|v

In [25]:
# Deduplicação caso aconteça
one_big_table_dedup = one_big_table.dropDuplicates()


one_big_table_dedup.createOrReplaceTempView("one_big_table_dedup")
one_big_table_dedup.cache()
one_big_table_dedup.count()

25/12/31 11:42:36 WARN CacheManager: Asked to cache already cached data.


1290526

In [30]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        ROUND(AVG(FPD) * 100, 2) as med_target_fpd
    FROM one_big_table_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(truncate=False)

+------+------------+-------------+--------------+
|SAFRA |total_linhas|cpf_distintos|med_target_fpd|
+------+------------+-------------+--------------+
|202410|203828      |203828       |22.69         |
|202411|227176      |227176       |24.88         |
|202412|227985      |227985       |23.92         |
|202501|221002      |221002       |23.61         |
|202502|203139      |203139       |22.37         |
|202503|207396      |207396       |23.76         |
+------+------------+-------------+--------------+



In [34]:
from delta.tables import DeltaTable

gold_path = "s3a://gold/one_big_table_publico/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, gold_path):
    print("Tabela silver não existe. Criando...")

    (
        one_big_table_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(gold_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_gold = DeltaTable.forPath(spark, gold_path)

    (
        delta_gold.alias("t")
        .merge(
            one_big_table_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA  = s.SAFRA 
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [35]:
spark.stop()